---
title: Trend-scanning labels
execute:
  enabled: true
---

`q.label.trend_scanning` fits forward linear regressions over several candidate bar horizons. For each event it keeps the slope with the largest absolute t-value and assigns a directional label when that statistic exceeds `min_t_value`.

A horizon of 5 means five bars after the event and therefore fits six prices including the event itself.

In [1]:
import pandas as pd

import qrt as q

close = q.data.datasets.load("spy")["close"]
close = close.loc[close.index.max() - pd.DateOffset(years=5) :]
volatility = close.pct_change(fill_method=None).ewm(span=20, min_periods=20).std()
events = q.label.cusum_filter(close, volatility.mul(0.75).where(volatility.gt(0)))
events = events[::5]
trends = q.label.trend_scanning(
    close,
    events=events,
    horizons=range(5, 21),
    min_t_value=2.0,
)
trends.tail()

,end_time,horizon,slope,t_value,return,label
event_time,,,,,,
2026-05-14,2026-06-04,14,0.002057,5.612657,0.011922,1
2026-05-26,2026-06-02,5,0.002688,8.716830,0.011964,1
2026-06-10,2026-07-10,20,0.000920,2.561758,0.043374,1
2026-06-23,2026-07-15,15,0.002054,7.111428,0.028940,1
2026-07-13,2026-07-20,5,-0.002491,-1.954247,-0.009450,0


## Selected horizons

The chosen `horizon`, `slope`, `t_value`, and realized return make the label auditable. `end_time` is the outcome boundary needed for leakage-aware splitting.

In [2]:
trends.groupby("label", observed=True).agg(
    observations=("label", "size"),
    median_horizon=("horizon", "median"),
    median_t_value=("t_value", "median"),
    average_return=("return", "mean"),
)

,observations,median_horizon,median_t_value,average_return
label,,,,
-1,48,14.0,-5.801710,-0.044580
0,3,15.0,-1.775394,0.002572
1,75,16.0,7.549171,0.038422


## Visualize selected trends

Markers show the selected trend direction at each event. Shaded intervals end at each regression's chosen horizon, so variable label lifetimes remain visible.

In [3]:
chart_start = close.index.max() - pd.DateOffset(years=1)
figure = q.plot.labels(
    close.loc[chart_start:],
    trends.loc[chart_start:],
    title="SPY trend-scanning labels (latest year)",
)
figure.show(renderer="notebook_connected")